In [ ]:
# =================================================
# cellules_sanguines.ipynb
# Classification des globules blancs
# =================================================

# 1. Import des librairies
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LinearRegression

# Config visuelle
plt.style.use('seaborn-v0_8')
sns.set_context('talk')

# Seeds pour reproductibilité
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow {tf.__version__}')
print('GPU(s) disponible(s) :', tf.config.list_physical_devices('GPU'))

# =================================================
# 2. Préparation des dossiers
# =================================================
# Modifie ce chemin si nécessaire
DATA_ROOT = Path('data') / 'raw'  # <- change si tes dossiers sont ailleurs
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'validation'
TEST_DIR = DATA_ROOT / 'test'

# Vérification des dossiers
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    print(f'{d} existe ? {d.exists()}')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # tu peux réduire à 16 ou 8 si trop lent

# =================================================
# 3. Chargement des datasets
# =================================================
base_train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
base_val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR, shuffle=False, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
base_test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, shuffle=False, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = base_train_ds.class_names
num_classes = len(class_names)
print(f'Classes détectées ({num_classes}) : {class_names}')

# =================================================
# 4. Prétraitement et augmentation
# =================================================
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.12),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.15),
], name='data_augmentation')

normalization_layer = layers.Rescaling(1.0 / 255)

def prepare_dataset(dataset, augment=False):
    def _process(images, labels):
        if augment:
            images = data_augmentation(images, training=True)
        images = normalization_layer(images)
        return images, labels
    return dataset.map(_process, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

train_ds = prepare_dataset(base_train_ds, augment=True)
val_ds = prepare_dataset(base_val_ds, augment=False)
test_ds = prepare_dataset(base_test_ds, augment=False)

# =================================================
# 5. Construction du CNN
# =================================================
def conv_block(x, filters, dropout_rate=0.2):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    return x

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = conv_block(inputs, 32, dropout_rate=0.1)
x = conv_block(x, 64, dropout_rate=0.15)
x = conv_block(x, 128, dropout_rate=0.2)
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs, name='wbc_cnn_classifier')
model.summary()

# =================================================
# 6. Compilation et entraînement
# =================================================
EPOCHS = 8  # <- changé pour 8 epochs
LEARNING_RATE = 1e-4

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_dir = Path('models') / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'wbc_cnn.keras'

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)

# =================================================
# 7. Courbes d'apprentissage
# =================================================
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# =================================================
# 8. Evaluation sur le test
# =================================================
test_loss, test_acc = model.evaluate(test_ds)
print(f'Loss test : {test_loss:.4f}')
print(f'Accuracy test : {test_acc:.4f}')

y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

# Matrice de confusion
conf_mat = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Classe réelle')
plt.xlabel('Prédiction')
plt.title('Matrice de confusion')
plt.tight_layout()
plt.show()

# Rapport de classification
report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print('Classification report:')
print(report)

# =================================================
# 9. Régression linéaire et résidus
# =================================================
y_prob_max = np.max(y_prob, axis=1)
y_true_flat = y_true.flatten()

reg = LinearRegression()
reg.fit(y_prob_max.reshape(-1,1), y_true_flat)
y_line = reg.predict(np.sort(y_prob_max).reshape(-1,1))

plt.figure(figsize=(7,5))
plt.scatter(y_prob_max, y_true_flat, alpha=0.5)
plt.plot(np.sort(y_prob_max), y_line, color='blue')
plt.xlabel('Score de confiance')
plt.ylabel('Classe réelle')
plt.title('Régression linéaire (Test)')
plt.tight_layout()
plt.show()

residus = y_true_flat - y_pred
plt.figure(figsize=(7,5))
plt.hist(residus, bins=10, color='steelblue', edgecolor='white')
plt.xlabel('Résidu')
plt.ylabel('Fréquence')
plt.title('Distribution des résidus')
plt.tight_layout()
plt.show()

# =================================================
# 10. Afficher quelques prédictions sur images de test
# =================================================
def show_sample_predictions(dataset, n_images=9):
    sample_batch = next(iter(dataset.unbatch().batch(n_images)))
    images, labels = sample_batch
    preds = model.predict(images)
    pred_labels = np.argmax(preds, axis=1)

    images = images.numpy()
    labels = labels.numpy()
    total = min(n_images, images.shape[0])

    cols = int(np.ceil(np.sqrt(total)))
    rows = int(np.ceil(total / cols))
    plt.figure(figsize=(3 * cols, 3 * rows))
    for idx in range(total):
        ax = plt.subplot(rows, cols, idx + 1)
        plt.imshow(images[idx])
        pred_name = class_names[pred_labels[idx]]
        true_name = class_names[labels[idx]]
        color = 'green' if pred_name == true_name else 'red'
        plt.title(f'P: {pred_name} T: {true_name}', color=color)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_sample_predictions(test_ds, n_images=9)

TensorFlow 2.21.0
GPU(s) disponible(s) : []
C:\Projet_IA\blood-cell-recognition-ml\data\raw\train existe ? True
C:\Projet_IA\blood-cell-recognition-ml\data\raw\validation existe ? True
C:\Projet_IA\blood-cell-recognition-ml\data\raw\test existe ? True
Found 11959 files belonging to 8 classes.
Found 2565 files belonging to 8 classes.
Found 2568 files belonging to 8 classes.
Classes détectées (8) : ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']


Model: "wbc_cnn_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 618,920 (2.36 MB)

 Trainable params: 617,512 (2.36 MB)

 Non-trainable params: 1,408 (5.50 KB)

Epoch 1/30
374/374 ━━━━━━━━━━━━━━━━━━━━ 0s 13s/step - accuracy: 0.5389 - loss: 1.3917 

KeyboardInterrupt: 